# Evaluate RAG with NeMo Retriever and RAGAS

This notebook walks through a complete retrieval-augmented generation (RAG) evaluation using **NeMo Retriever Library**, **FinanceBench or your own dataset**, and **RAGAS**. It builds a LanceDB index, retrieves evidence, generates answers, scores the results, and presents a reusable analysis.

## What you will measure

- **Document nDCG@10** — how highly the relevant source documents rank among the top 10 retrieved results after duplicate chunks are collapsed.
- **Answer Accuracy** — how closely the generated answer agrees with the reference answer.
- **Context Relevance** — whether the retrieved passages are useful for answering the question.
- **Response Groundedness** — whether the generated answer is supported by the retrieved passages.

## Workflow

1. Install dependencies.
2. Choose FinanceBench or configure a custom dataset.
3. Validate the corpus and ground truth.
4. Build or reuse a LanceDB index.
5. Retrieve contexts and measure document nDCG@10.
6. Generate answers and evaluate them with RAGAS.
7. Analyze the results.

> **Before you start:** Run this notebook from the repository `examples` directory. Choose exactly one dataset option. Hosted ingestion, generation, and judging require an NVIDIA API key.


## 1. Install Dependencies

Install NeMo Retriever, RAGAS, the NVIDIA LangChain integration, and the OpenAI SDK into the active notebook kernel. Run this cell once when setting up the environment, then restart the kernel before continuing. The `ipykernel<7` compatibility requirement avoids a known `nest_asyncio` context collision in RAGAS 0.3.2.


In [ ]:
%pip install -qe "../nemo_retriever[llm]" "ragas==0.3.2" "pyarrow<21" "langchain-community==0.4.1" langchain-nvidia-ai-endpoints openai "ipykernel<7"


## 2. Configure the Dataset

Choose **one** option below. The last dataset configuration cell you run determines which corpus and ground-truth records the rest of the notebook uses.

A dataset needs two inputs:

- A corpus directory containing documents supported by NeMo Retriever.
- A CSV, JSON, or JSONL file containing questions and reference answers.


### Option A: Download and Use FinanceBench

FinanceBench is the ready-to-run example and the default configuration for this notebook. Clone the dataset, then run the configuration cell that points to its PDFs and open-source question-answer split.

> If `../data/financebench` already exists, skip the clone cell and run only the configuration cell.


In [ ]:
!git clone https://github.com/patronus-ai/financebench.git ../data/financebench

In [ ]:
from pathlib import Path

dataset_config = {
    "name": "financebench",
    "corpus_dir": Path("../data/financebench/pdfs"),
    "ground_truth_path": Path(
        "../data/financebench/data/financebench_open_source.jsonl"
    ),
    "ground_truth_format": "jsonl",
    "id_field": "financebench_id",
    "query_field": "question",
    "answer_field": "answer",
    "document_field": "doc_name",
}    

### Option B: Bring Your Own Dataset

Use this option to evaluate another NeMo Retriever-compatible corpus. Edit the paths and field names in the template cell, then run it instead of the FinanceBench configuration cell.

`query_field` and `answer_field` are required. Set `id_field` or `document_field` to `None` when your ground truth does not contain them. Without `document_field`, the notebook still runs RAGAS evaluation but skips document-level nDCG@10.


In [ ]:
from pathlib import Path

# Edit these values, then run this cell instead of Option A.
dataset_config = {
    # Used for the LanceDB table and output directory names.
    "name": "my_dataset",

    # Directory containing the documents to ingest.
    "corpus_dir": Path("../data/my_dataset/corpus"),

    # CSV, JSON, or JSONL file containing questions and reference answers.
    "ground_truth_path": Path("../data/my_dataset/ground_truth.jsonl"),
    "ground_truth_format": "jsonl",

    # Column or object-key mappings in the ground-truth records.
    # Set optional fields to None when they are unavailable.
    "id_field": None,
    "query_field": "question",
    "answer_field": "answer",
    "document_field": None,
}


## 3. Validate the Dataset

Run this section after choosing a dataset. It checks that the corpus and ground-truth file exist, loads CSV/JSON/JSONL records, verifies the configured fields, and rejects empty questions or answers before any expensive work begins.

A successful run prints the dataset name, corpus file count, ground-truth record count, and one example question. Fix any assertion here before continuing.


In [ ]:
import csv
import json

corpus_dir = Path(dataset_config["corpus_dir"])
ground_truth_path = Path(dataset_config["ground_truth_path"])
ground_truth_format = dataset_config["ground_truth_format"].lower()

assert corpus_dir.is_dir(), f"Corpus directory not found: {corpus_dir}"
assert ground_truth_path.is_file(), f"Ground-truth file not found: {ground_truth_path}"

corpus_files = sorted(path for path in corpus_dir.rglob("*") if path.is_file())
assert corpus_files, f"No files found in corpus directory: {corpus_dir}"

if ground_truth_format == "jsonl":
    with ground_truth_path.open(encoding="utf-8") as file:
        ground_truth_records = [json.loads(line) for line in file if line.strip()]
elif ground_truth_format == "json":
    with ground_truth_path.open(encoding="utf-8") as file:
        ground_truth_records = json.load(file)
    assert isinstance(ground_truth_records, list), "Ground-truth JSON must contain a list of records."
elif ground_truth_format == "csv":
    with ground_truth_path.open(encoding="utf-8", newline="") as file:
        ground_truth_records = list(csv.DictReader(file))
else:
    raise ValueError("ground_truth_format must be 'csv', 'json', or 'jsonl'.")

assert ground_truth_records, f"No records found in {ground_truth_path}"

configured_fields = {
    name: dataset_config.get(name)
    for name in ("id_field", "query_field", "answer_field", "document_field")
    if dataset_config.get(name)
}
missing_fields = {
    field
    for field in configured_fields.values()
    if any(field not in record for record in ground_truth_records)
}
assert not missing_fields, f"Missing configured ground-truth fields: {sorted(missing_fields)}"

query_field = dataset_config["query_field"]
answer_field = dataset_config["answer_field"]
assert all(str(record[query_field]).strip() for record in ground_truth_records), "Found an empty question."
assert all(str(record[answer_field]).strip() for record in ground_truth_records), "Found an empty reference answer."

print(f"Dataset: {dataset_config['name']}")
print(f"Corpus files: {len(corpus_files)}")
print(f"Ground-truth records: {len(ground_truth_records)}")
print(f"Example question: {ground_truth_records[0][query_field]}")


## 4. Configure and Ingest with NeMo Retriever

Configure the LanceDB location, embedding model, retrieval depth, and evaluation sample size. `MAX_QUESTIONS=50` keeps the first run manageable; set it to `None` for the full ground-truth split.

The ingest cell uses the Ray batch CLI. Set `REBUILD_INDEX=True` to create or replace the table, or set it to `False` to reuse an existing index. On systems without local inference, embedding uses the NVIDIA hosted endpoint and requires `NVIDIA_API_KEY`.

> **Ingestion can take a while:** small smoke tests may finish in minutes, but a corpus such as FinanceBench can take tens of minutes or several hours depending on extraction, model downloads, endpoint speed, and hardware. Some stages may produce little output, and the LanceDB table may not appear until near the end. Avoid interrupting a busy kernel unless you have confirmed that the process is stalled.


In [ ]:
# NRL index configuration
LANCEDB_URI = "lancedb-" + dataset_config["name"]
TABLE_NAME = dataset_config["name"]
EMBED_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"

# Evaluation configuration
MAX_QUESTIONS = 50  # Set to None to evaluate every question.
NDCG_K = 10
RETRIEVAL_K = 10

# Set to True to create or replace the LanceDB index.
REBUILD_INDEX = False

assert RETRIEVAL_K >= NDCG_K, (
    f"RETRIEVAL_K must be at least {NDCG_K} to calculate nDCG@{NDCG_K}."
)


In [ ]:
import os
import sys
from getpass import getpass

os.environ["PATH"] = (
    os.path.dirname(sys.executable)
    + os.pathsep
    + os.environ["PATH"]
)

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA API key: ")

if REBUILD_INDEX:
    !retriever ingest batch "{corpus_dir}" \
        --lancedb-uri "{LANCEDB_URI}" \
        --table-name "{TABLE_NAME}" \
        --embed-model-name "{EMBED_MODEL}" \
        --overwrite
else:
    print(f"Reusing table {TABLE_NAME!r} from {LANCEDB_URI!r}")


## 5. Retrieve Contexts and Measure nDCG@10

Retrieve the top `RETRIEVAL_K` passages for each selected question using the same embedding model and LanceDB table created during ingestion. The resulting dataframe retains the question, reference answer, retrieved text, and retrieval metadata for later stages.

When `document_field` is configured, this section reports document-level nDCG@10. Ground-truth documents are treated as binary relevant items, and duplicate chunks from the same document are collapsed within the top 10 results. A score of 1.0 means the relevant documents occupy the best possible ranks in the top 10; lower scores indicate that they appeared later or were missed.


In [ ]:
import pandas as pd
from nemo_retriever.graph.retriever import Retriever

selected_records = (
    ground_truth_records
    if MAX_QUESTIONS is None
    else ground_truth_records[:MAX_QUESTIONS]
)
questions = [
    record[dataset_config["query_field"]]
    for record in selected_records
]

retriever = Retriever(
    vdb_kwargs={
        "uri": LANCEDB_URI,
        "table_name": TABLE_NAME,
    },
    embed_kwargs={
        "model_name": EMBED_MODEL,
        "embed_model_name": EMBED_MODEL,
    },
    top_k=RETRIEVAL_K,
)

retrieval_results = retriever.retrieve_batch(
    questions,
    top_k=RETRIEVAL_K,
)

print(f"Retrieved contexts for {len(retrieval_results)} questions")


In [ ]:
import math


def normalise_document_name(value):
    """Normalize a ground-truth or retrieved document identifier."""
    text = str(value or "").strip()
    if not text:
        return ""
    return Path(text).stem.casefold()


def source_document_name(metadata):
    """Extract the source document name from NRL retrieval metadata."""
    for key in ("source_id", "source", "path", "source_path"):
        value = metadata.get(key)
        if not value:
            continue

        if isinstance(value, dict):
            nested = source_document_name(value)
            if nested:
                return nested

        if isinstance(value, str) and value.lstrip().startswith("{"):
            try:
                parsed = json.loads(value)
            except json.JSONDecodeError:
                parsed = None
            if isinstance(parsed, dict):
                nested = source_document_name(parsed)
                if nested:
                    return nested

        return normalise_document_name(value)

    return ""


def gold_document_names(value):
    values = value if isinstance(value, (list, tuple, set)) else [value]
    return {
        normalised
        for item in values
        if (normalised := normalise_document_name(item))
    }


def document_ndcg_at_k(metadata, gold_documents, k):
    """Calculate binary document-level nDCG after deduplicating retrieved chunks."""
    gold = gold_document_names(gold_documents)
    if not gold:
        return None

    ranked_documents = []
    seen = set()
    for hit in metadata[:k]:
        document = source_document_name(hit)
        if document and document not in seen:
            seen.add(document)
            ranked_documents.append(document)

    dcg = sum(
        1 / math.log2(rank + 1)
        for rank, document in enumerate(ranked_documents, start=1)
        if document in gold
    )
    ideal_relevant = min(len(gold), k)
    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )
    return dcg / idcg if idcg else None


id_field = dataset_config.get("id_field")
query_field = dataset_config["query_field"]
answer_field = dataset_config["answer_field"]
document_field = dataset_config.get("document_field")
ndcg_column = f"ndcg_at_{NDCG_K}"

retrieval_rows = []
for index, (record, result) in enumerate(
    zip(selected_records, retrieval_results),
    start=1,
):
    gold_document = record.get(document_field) if document_field else None
    retrieval_rows.append({
        "query_id": record.get(id_field, index) if id_field else index,
        "question": record[query_field],
        "reference": record[answer_field],
        "gold_document": gold_document,
        "retrieved_contexts": result.chunks,
        "retrieval_metadata": result.metadata,
        ndcg_column: document_ndcg_at_k(
            result.metadata,
            gold_document,
            NDCG_K,
        ),
    })

retrieval_df = pd.DataFrame(retrieval_rows)

if document_field:
    display(retrieval_df[[ndcg_column]].mean().to_frame("mean_score"))
else:
    print("No document_field configured; nDCG@10 was skipped.")

display(retrieval_df.head())


## 6. Generate Answers

Generate one answer per question with the NVIDIA OpenAI-compatible chat endpoint. Each request includes only the question and the contexts retrieved in Section 5, so generation does not perform another retrieval pass.

The API key is requested with `getpass` and is not written into the notebook. Progress messages distinguish attempted requests from successful answers, and failures are stored in `generation_error` for inspection.

> Hosted models can be rate limited. Start with a small `MAX_QUESTIONS`, and inspect the representative errors printed by this section before retrying a large run.


In [ ]:
import os
import random
import time
from getpass import getpass

from openai import APIConnectionError, APITimeoutError, OpenAI

GENERATOR_MODEL = "nvidia/nemotron-3-super-120b-a12b"
GENERATOR_MAX_TOKENS = 4096
GENERATOR_TEMPERATURE = 0.0
GENERATOR_ENABLE_THINKING = True
GENERATOR_DEBUG_ERRORS = True
GENERATOR_MAX_ATTEMPTS = 5
GENERATOR_RETRY_BASE_SECONDS = 2.0
GENERATOR_RETRY_MAX_SECONDS = 30.0

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA API key: ")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
    max_retries=0,  # Retries are handled visibly in the next cell.
)


In [ ]:
def generation_messages(question, contexts):
    numbered_contexts = "\n\n".join(
        f"[{index}] {context}"
        for index, context in enumerate(contexts, start=1)
    )
    return [
        {
            "role": "system",
            "content": (
                "Answer the question using only the retrieved context. "
                "Give a concise, direct answer. If the context does not "
                "contain the answer, say that the answer is not available."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question}\n\n"
                f"Retrieved context:\n{numbered_contexts}"
            ),
        },
    ]


def format_generation_error(error):
    details = [f"{type(error).__name__}: {error}"]
    status_code = getattr(error, "status_code", None)
    request_id = getattr(error, "request_id", None)
    if status_code is not None:
        details.append(f"HTTP status: {status_code}")
    if request_id:
        details.append(f"request_id: {request_id}")
    return " | ".join(details)


def is_retryable_generation_error(error):
    status_code = getattr(error, "status_code", None)
    return (
        isinstance(error, (APIConnectionError, APITimeoutError))
        or status_code in {408, 409, 429, 500, 502, 503, 504}
    )


responses = []
generation_errors = []

for position, (_, row) in enumerate(retrieval_df.iterrows(), start=1):
    query_id = row.get("query_id", position)
    answer = None
    final_error = None

    for attempt in range(1, GENERATOR_MAX_ATTEMPTS + 1):
        try:
            completion = client.chat.completions.create(
                model=GENERATOR_MODEL,
                messages=generation_messages(
                    row["question"],
                    row["retrieved_contexts"],
                ),
                temperature=GENERATOR_TEMPERATURE,
                max_tokens=GENERATOR_MAX_TOKENS,
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": GENERATOR_ENABLE_THINKING
                    }
                },
            )
            answer = completion.choices[0].message.content
            if not answer or not answer.strip():
                raise ValueError("The model returned an empty answer.")
            answer = answer.strip()
            break
        except Exception as error:
            final_error = error
            should_retry = (
                is_retryable_generation_error(error)
                and attempt < GENERATOR_MAX_ATTEMPTS
            )
            if not should_retry:
                break

            delay = min(
                GENERATOR_RETRY_MAX_SECONDS,
                GENERATOR_RETRY_BASE_SECONDS * (2 ** (attempt - 1)),
            ) + random.uniform(0, 1)
            print(
                f"Query {query_id}: attempt {attempt}/"
                f"{GENERATOR_MAX_ATTEMPTS} failed with "
                f"{format_generation_error(error)}. "
                f"Retrying in {delay:.1f}s..."
            )
            time.sleep(delay)

    if answer is not None:
        responses.append(answer)
        generation_errors.append(None)
    else:
        responses.append("")
        error_message = format_generation_error(final_error)
        generation_errors.append(error_message)
        if GENERATOR_DEBUG_ERRORS:
            print(
                f"\nGeneration failed for query {query_id} "
                f"(row {position}) after {attempt} attempt(s):\n"
                f"{error_message}\n"
            )

    if position % 10 == 0 or position == len(retrieval_df):
        successful = sum(error is None for error in generation_errors)
        print(
            f"Attempted {position}/{len(retrieval_df)} answers; "
            f"{successful} successful."
        )

retrieval_df["response"] = responses
retrieval_df["generation_error"] = generation_errors

evaluation_df = retrieval_df[
    retrieval_df["generation_error"].isna()
].reset_index(drop=True)

if evaluation_df.empty:
    unique_errors = retrieval_df["generation_error"].dropna().unique()
    print("Representative generation errors:")
    for error in unique_errors[:5]:
        print(f"- {error}")
    raise RuntimeError("Every answer-generation request failed.")

ragas_records = evaluation_df[
    ["question", "retrieved_contexts", "response", "reference"]
] .rename(columns={"question": "user_input"}).to_dict("records")

print(
    f"Generated {len(evaluation_df)} successful answers; "
    f"{len(retrieval_df) - len(evaluation_df)} failed."
)

failed_generations = retrieval_df.loc[
    retrieval_df["generation_error"].notna(),
    ["query_id", "question", "generation_error"],
]
if not failed_generations.empty:
    print(f"Failure details ({len(failed_generations)}):")
    display(failed_generations)


## 7. Evaluate with RAGAS

Use an NVIDIA-hosted judge model to score every successful answer with three complementary metrics:

- **Answer Accuracy** compares the response with the reference answer.
- **Context Relevance** measures whether the retrieved passages address the question.
- **Response Groundedness** checks whether claims in the response are supported by those passages.

`RAGAS_MAX_WORKERS=1` intentionally limits concurrency to reduce public-endpoint rate-limit errors. A self-hosted compatible judge can be substituted when higher throughput is needed.


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerAccuracy, ContextRelevance, ResponseGroundedness
from ragas.run_config import RunConfig

JUDGE_MODEL = "openai/gpt-oss-120b"
RAGAS_MAX_WORKERS = 1
RAGAS_MAX_WAIT = 120

judge_llm = ChatNVIDIA(model=JUDGE_MODEL)
evaluation_dataset = EvaluationDataset.from_list(ragas_records)

ragas_results = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        AnswerAccuracy(),
        ContextRelevance(),
        ResponseGroundedness(),
    ],
    llm=LangchainLLMWrapper(judge_llm),
    run_config=RunConfig(
        max_workers=RAGAS_MAX_WORKERS,
        max_wait=RAGAS_MAX_WAIT,
    ),
)

ragas_results


## 8. Analyze Results

Combine document nDCG@10, generation status, and per-question RAGAS scores into one report. This section displays the mean retrieval and RAGAS scores followed by a sample of the row-level results.


In [ ]:
ragas_df = ragas_results.to_pandas().reset_index(drop=True)

report_columns = [
    "query_id",
    "gold_document",
    ndcg_column,
    "generation_error",
]
report_df = pd.concat(
    [
        evaluation_df[report_columns].reset_index(drop=True),
        ragas_df,
    ],
    axis=1,
)

ragas_metric_columns = list(
    ragas_df.select_dtypes(include="number").columns
)
metric_columns = [
    column
    for column in [ndcg_column, *ragas_metric_columns]
    if column in report_df.columns
]

print("Mean evaluation scores:")
display(
    report_df[metric_columns]
    .mean()
    .to_frame("mean_score")
)

print("Per-question results:")
display(report_df.head())
